# Analysis

## Setting parameters

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from quickrules.data_handling import calculate_score, count_all_rules, count_all_attributes, bold
from pathlib import Path
from quickrules.data_handling import balanced_accuracy_score
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np

In [3]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass',   
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     # 'winequality-red',  # todo temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
    'ecoli',
    'wisconsin',
    'dermatology',
    'cleveland',
    'spectfheart',
    'bupa',
    'yeast',
    'heart',
    'australian',
    'glass',
    'pima',
]

In [42]:
data_sets = ['ecoli']  # inclusion_test

In [43]:
len(data_sets)

1

In [44]:
data_base = Path.cwd() / 'keel-data'
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [45]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [46]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [47]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [48]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )

## Adding results for non rule based models


In [49]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [50]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [51]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in data_sets}

## FRRI Models

In [52]:
frri_models = {
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    # 'lower-new-check' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / results_base / 'lower-approx-luka-impl-incl.99'
    'incl 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "1-1e-6",
    'incl 1e-4' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    'incl 1e-3' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "999",
    'incl 1e-2' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "99",
    # 'incl 0.95' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "95",
    'owa 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'linear-owa-inclusion' / '1-1e-6',
    'quickreduct 1e-6': Path.cwd() / results_base / 'lower-approx' / 'quickreduct-ordering' / '1-1e-6',
}

In [53]:
for model, path in frri_models.items():
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

## Checking all of the models

In [54]:
scores.keys()

dict_keys(['modlem', 'furia', 'ripper', 'qr', 'naive-bayes', 'svm', 'tree', 'frnn', 'incl 1e-6', 'incl 1e-4', 'incl 1e-3', 'incl 1e-2', 'owa 1e-6', 'quickreduct 1e-6'])

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [55]:
names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    'frri-paper': [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
        'furia',
        'ripper',
    ],
    'inclusion' : [
        'incl 1e-6',
        'incl 1e-4',
        'incl 1e-3', 
        'incl 1e-2', 
        # 'incl 0.95',
        'owa 1e-6',
        'quickreduct 1e-6',
    ]
}

In [56]:
nr = 'inclusion' # 6

In [57]:
main_folder = 'tables_inclusion_set2' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [58]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])

In [59]:
for model in names[nr]:
    for data_set in data_sets:
    # for data_set, score in scores[model].items():
        table_acc.loc[data_set, model] = scores[model][data_set]

In [60]:
table_acc['max'] = table_acc.max(axis='columns', skipna=True)

In [61]:
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)

In [62]:
table_acc

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,owa 1e-6,quickreduct 1e-6,max
ecoli,0.700635,0.720659,0.69504,0.681103,0.700635,0.699771,0.720659
mean,0.700635,0.720659,0.69504,0.681103,0.700635,0.699771,0.720659


In [35]:
bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_54692/3326315922.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [63]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, rule_count in rules[model].items():
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]  #  rule_count
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)

In [64]:
table_rule

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,owa 1e-6,quickreduct 1e-6
ecoli,54.8,56.8,54.2,41.8,54.8,54.6
mean,54.8,56.8,54.2,41.8,54.8,54.6


In [38]:
bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_54692/3026783303.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [65]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    # for data_set, attribute_count in attributes[model].items():
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]  # attribute_count
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)

In [66]:
table_attribute

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,owa 1e-6,quickreduct 1e-6
ecoli,3.866034,3.854646,3.871903,3.764987,3.867921,3.93786
mean,3.866034,3.854646,3.871903,3.764987,3.867921,3.93786


In [41]:
bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_54692/3212901870.py:2: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [28]:
from scipy import stats
import scikit_posthocs as sph

In [37]:
subject = table_acc

In [38]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('qr', axis, inplace=True)

In [39]:
no_mean

,lower-new-check,modlem,qr,furia,ripper
australian,0.787517,0.861,0.865348,0.8645,0.852
bands,0.675141,0.6035,0.533045,0.6295,0.5815
bupa,0.600724,0.6525,0.5,0.6575,0.6535
cleveland,0.290397,0.2764,0.208347,0.2452,0.2384
dermatology,0.895363,0.941333,0.521995,0.95,0.94183
ecoli,0.720659,0.512,0.17585,0.529875,0.54475
glass,0.679067,0.518333,0.31381,0.521857,0.494857
heart,0.7675,0.7605,0.785833,0.818,0.7625
ionosphere,0.912468,0.8905,0.658974,0.887,0.892
pima,0.687633,0.6985,0.502775,0.7025,0.6985


In [32]:
stats.wilcoxon(no_mean['lower-new-check'], no_mean['furia'], alternative="greater")

WilcoxonResult(statistic=171.0, pvalue=3.814697265625e-06)

In [33]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=53.48100477475607, pvalue=1.1983286540382202e-05)

In [34]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.406796,9.410312e-03,0.051265,7.314025e-04
modlem,0.406796,1.000000,5.744072e-02,0.006980,4.034528e-05
qr,0.009410,0.057441,1.000000e+00,0.000002,3.228391e-09
furia,0.051265,0.006980,1.768152e-06,1.000000,2.443989e-01
ripper,0.000731,0.000040,3.228391e-09,0.244399,1.000000e+00


In [35]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

qr                    0.0
lower-new-check    1142.0
modlem             1164.6
furia              1512.6
ripper             1526.6
dtype: object

In [40]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
furia           &  2.277778 \\
lower-new-check &  2.555556 \\
modlem          &  2.722222 \\
ripper          &  3.000000 \\
qr              &  4.444444 \\
\bottomrule
\end{tabular}


/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_74098/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [152]:
ranks['lower-new-check'].value_counts()

1.0    7
4.0    6
3.0    2
2.0    2
5.0    1
Name: lower-new-check, dtype: int64

In [153]:
ranks.apply(pd.value_counts)

,lower-new-check,modlem,qr,furia,ripper
1.0,7.0,4,1.0,5.0,1
2.0,2.0,2,1.0,7.0,5
2.5,NaN,1,NaN,NaN,1
3.0,2.0,6,1.0,3.0,4
3.5,NaN,1,NaN,NaN,1
4.0,6.0,3,1.0,2.0,5
5.0,1.0,1,14.0,1.0,1


On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000
